# Day 044 — Exercise 3: get_items

**What you'll build:** `get_items(session, category=None) -> list[Item]` — query all items, or filter by category when provided.

**Why it matters:** `select(Item)` is the SQLAlchemy 2.x ORM query. `.where(Item.category == category)` appends a WHERE clause using Python attribute access — no SQL string needed. `session.execute(stmt).scalars().all()` converts the result set into a list of ORM objects, each with full attribute access.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
from sqlalchemy import create_engine, String, Float, Integer, select
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, Session
from sqlalchemy.pool import StaticPool


class Base(DeclarativeBase):
    pass


class Item(Base):
    __tablename__ = 'items'
    id:       Mapped[int]   = mapped_column(primary_key=True)
    name:     Mapped[str]   = mapped_column(String(100))
    category: Mapped[str]   = mapped_column(String(50))
    price:    Mapped[float] = mapped_column()
    quantity: Mapped[int]   = mapped_column(default=0)

    def __repr__(self):
        return f'Item(id={self.id}, name={self.name!r}, price={self.price})'


def setup_engine(url='sqlite:///:memory:'):
    engine = create_engine(
        url,
        connect_args={'check_same_thread': False},
        poolclass=StaticPool,
    )
    Base.metadata.create_all(engine)
    return engine


def add_item(session, name, category, price, quantity=0):
    item = Item(name=name, category=category, price=price, quantity=quantity)
    session.add(item)
    session.commit()
    session.refresh(item)
    return item


engine = setup_engine()
session = Session(engine)

# Seed data
add_item(session, 'Laptop',    'Electronics', 999.99, 5)
add_item(session, 'Headphones','Electronics', 149.99, 12)
add_item(session, 'Desk Chair','Furniture',   349.00, 3)
add_item(session, 'Bookcase',  'Furniture',   199.00, 8)
add_item(session, 'Pen Set',   'Stationery',   12.99, 50)

## Your Implementation

In [ ]:
def get_items(session, category=None):
    """
    Return all items, optionally filtered by category.

    Use select(Item) as the base statement.
    Append .where(Item.category == category) if category is not None.
    Execute with session.execute(stmt).scalars().all().
    Return a list.
    """
    # TODO: stmt = select(Item)
    # TODO: if category is not None:
    # TODO:     stmt = stmt.where(Item.category == category)
    # TODO: return list(session.execute(stmt).scalars().all())
    pass

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    # Check 1: defined
    try:
        assert 'get_items' in globals()
        passed += 1; print('\u2705 Check 1: get_items is defined')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: returns a list
    try:
        result = get_items(session)
        assert isinstance(result, list), \
            f'expected list, got {type(result).__name__}'
        passed += 1; print('\u2705 Check 2: returns a list')
    except Exception as e:
        print(f'\u274c Check 2: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 3: no filter returns all 5 seeded items
    try:
        assert len(result) == 5, \
            f'expected 5 items, got {len(result)}'
        assert all(isinstance(i, Item) for i in result)
        passed += 1; print('\u2705 Check 3: no filter returns all 5 items')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: category filter returns only matching items
    try:
        elec = get_items(session, category='Electronics')
        assert len(elec) == 2, \
            f'expected 2 Electronics, got {len(elec)}'
        assert all(i.category == 'Electronics' for i in elec)
        passed += 1; print('\u2705 Check 4: category=Electronics returns 2 items')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: non-existent category returns empty list
    try:
        none_found = get_items(session, category='NonExistent')
        assert none_found == [], \
            f'expected [], got {none_found}'
        passed += 1; print('\u2705 Check 5: non-existent category returns []')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
def get_items(session, category=None):
    stmt = select(Item)
    if category is not None:
        stmt = stmt.where(Item.category == category)
    return list(session.execute(stmt).scalars().all())
```

</details>